# 1. Análisis exploratorio de la serie BTCUSDT

Analizamos el precio de cierre diario del par BTCUSDT en Binance entre el 1 de enero de 2018 y el 31 de diciembre de 2025. El objetivo es caracterizar la serie antes de modelarla: nivel y tendencia, distribución de los retornos, volatilidad, estacionariedad y dependencia temporal. Al final del notebook marcamos el tramo que reservamos como hold-out temporal.

In [ ]:
# Rutas: en Colab se usa la carpeta de Drive; en el repositorio, la raíz del proyecto.
import os, glob, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DRIVE = Path("/content/drive/MyDrive/tarea2_series")
if DRIVE.exists():
    BASE, FIGS = DRIVE, DRIVE / "figs"
else:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    FIGS = BASE / "notebooks" / "figs"
RESULTS = BASE / os.environ.get("T2_RESULTS_DIR", "results")
FIGS.mkdir(parents=True, exist_ok=True)

LAGS, HS, ESTRATEGIAS = [7, 14, 21, 28], [7, 14, 21, 28], ["gk", "fc", "kf"]
NOMBRE = {"gk": "Group K-Fold", "fc": "Forward Chaining", "kf": "K-Fold"}
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # orden categorico fijo
COL_EST = dict(zip(ESTRATEGIAS, COL))
COL_LAG = dict(zip(LAGS, COL))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "axes.prop_cycle": plt.cycler(color=COL), "lines.linewidth": 1.6,
                     "legend.frameon": False, "axes.titlesize": 11, "axes.labelsize": 10})

df = pd.read_csv(BASE / "data" / "btc_1d.csv", parse_dates=["date"])
y = df["close"].to_numpy(float)
DATES = df["date"].to_numpy()
N_HOLD = max(int(len(y) * 0.10), 56)
N_CV = len(y) - N_HOLD


def guardar(fig, nombre):
    fig.savefig(FIGS / f"{nombre}.png", bbox_inches="tight", dpi=150)

## 1.1 Datos

In [ ]:
print(f"Observaciones: {len(df)} días | desde {df.date.min().date()} hasta {df.date.max().date()}")
print("Días faltantes:", int((df.date.diff().dt.days > 1).sum()), "| valores nulos:", int(df.isna().sum().sum()))
print(f"Hold-out temporal: últimos {N_HOLD} días ({df.date.iloc[N_CV].date()} a {df.date.iloc[-1].date()})")
df.describe().T.round(2)

## 1.2 Nivel de precio y tendencia

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for a, logy in zip(ax, (False, True)):
    a.plot(df.date, df.close, color=COL[0], lw=1.2, label="Cierre diario")
    a.axvspan(df.date.iloc[N_CV], df.date.iloc[-1], color="0.85", alpha=0.6, lw=0, label="Hold-out")
    a.set_ylabel("USD (escala log)" if logy else "USD")
    if logy:
        a.set_yscale("log")
ax[0].set_title("BTCUSDT: precio de cierre diario (2018–2025)")
ax[0].legend(loc="upper left")
guardar(fig, "eda_precio"); plt.show()

## 1.3 Retornos logarítmicos

In [ ]:
r = np.diff(np.log(y))
from scipy import stats
print(f"media {r.mean():.5f} | desv. estándar {r.std():.4f} | asimetría {stats.skew(r):.2f} | "
      f"curtosis en exceso {stats.kurtosis(r):.2f}")
print(f"Jarque-Bera p = {stats.jarque_bera(r).pvalue:.2e}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(df.date[1:], r, color=COL[0], lw=0.6)
ax[0].set_title("Retorno logarítmico diario"); ax[0].set_ylabel("log(P_t / P_t-1)")
ax[1].hist(r, bins=120, density=True, color=COL[0], alpha=0.75, label="Observado")
xx = np.linspace(r.min(), r.max(), 400)
ax[1].plot(xx, stats.norm.pdf(xx, r.mean(), r.std()), color=COL[1], label="Normal ajustada")
ax[1].set_title("Distribución de los retornos"); ax[1].legend()
guardar(fig, "eda_retornos"); plt.show()

## 1.4 Volatilidad

In [ ]:
vol = pd.Series(r, index=df.date[1:]).rolling(30).std() * np.sqrt(365)
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(vol.index, vol, color=COL[0], lw=1.2)
ax.set_title("Volatilidad anualizada (ventana móvil de 30 días)"); ax.set_ylabel("Volatilidad")
guardar(fig, "eda_volatilidad"); plt.show()
print(vol.groupby(vol.index.year).mean().round(3).to_string())

## 1.5 Estacionariedad (ADF y KPSS)

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
filas = []
for nombre, s in (("precio", y), ("log(precio)", np.log(y)), ("retorno log", r)):
    adf_p = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    filas.append(dict(serie=nombre, ADF_pvalor=adf_p, KPSS_pvalor=kpss_p))
tabla_est = pd.DataFrame(filas).round(4)
tabla_est.to_csv(RESULTS / "eda_estacionariedad.csv", index=False)
tabla_est

ADF tiene como hipótesis nula una raíz unitaria (no estacionaria); KPSS tiene como hipótesis nula la estacionariedad. KPSS reporta p-valores acotados al intervalo [0.01, 0.10].

## 1.6 Dependencia temporal (ACF y PACF)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, ax = plt.subplots(2, 2, figsize=(11, 6))
plot_acf(np.log(y), lags=60, ax=ax[0, 0], title="ACF de log(precio)")
plot_pacf(np.log(y), lags=60, ax=ax[0, 1], title="PACF de log(precio)", method="ywm")
plot_acf(r, lags=60, ax=ax[1, 0], title="ACF del retorno log")
plot_acf(r ** 2, lags=60, ax=ax[1, 1], title="ACF del retorno log al cuadrado")
fig.tight_layout(); guardar(fig, "eda_acf"); plt.show()

## 1.7 Volumen negociado

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(df.date, df.volume.rolling(30).mean(), color=COL[0], lw=1.2)
ax.set_title("Volumen diario (media móvil de 30 días, BTC)"); ax.set_ylabel("BTC")
guardar(fig, "eda_volumen"); plt.show()